# Synthetic Tabular Data Generation - Staged Optimization Driver

**Enhanced notebook for synthetic tabular data generation with staged hyperparameter optimization.**

This notebook provides a staged optimization workflow with:
- Section 1: Setup and imports
- Section 2: Data loading, preprocessing, and EDA
- Section 3: Demo models with default parameters
- Section 4: **Staged Hyperparameter Optimization** (NEW)
  - 4.1: Configuration
  - 4.2: Pilot phase (15 trials) with time estimates
  - 4.3: Continuation options
  - 4.4: Diminishing returns analysis
  - 4.5: Additional batches (optional)
- Section 5: Final models with best parameters

Based on STG-Driver-breast-cancer.ipynb with staged optimization enhancements.

## 1 Setup and Configuration

In [1]:
# Code Chunk ID: CHUNK_1_0_0_A - Import Setup Module
# Import all functionality from setup.py
import sys
sys.path.insert(0, "/home/ec2-user/SageMaker/tableGenCompare")

from setup import *

# Refresh session timestamp to current date
refresh_session_timestamp()

# ============================================================================
# FRESH START CONTROL - Set to True to wipe all checkpoints and start over
# ============================================================================
FRESH_START = True   # <-- Change to True to clear ALL saved progress
# ============================================================================

print("SETUP MODULE IMPORTED SUCCESSFULLY!")
print(f"FRESH_START = {FRESH_START}")
print("=" * 60)

[OK] Essential data science libraries imported successfully!
[OK] Model registry loaded from src/models/registry.py
[OK] Batch training module loaded from src/models/batch_training.py
[OK] Search spaces module loaded from src/models/search_spaces.py
[OK] Batch optimization module loaded from src/models/batch_optimization.py
[OK] Staged optimization module loaded from src/models/staged_optimization.py
[CONFIG] Session timestamp: 2026-03-25
[OK] Parameter management functions loaded from src/utils/parameters.py
[OK] Hyperparameter optimization evaluation functions loaded from src/evaluation/hyperparameters.py
[OK] Optuna objective functions for PATE-GAN and MEDGAN loaded (Phase 5)
Detected sklearn 1.7.2 - applying compatibility patch...
INFO: Applying sklearn compatibility patches for version 1.7.2
Global sklearn compatibility patch applied successfully
CTAB-GAN imported successfully
[OK] CTAB-GAN+ detected and available
[OK] GANerAidModel imported successfully from src.models.implementa

## 2 Data Loading and Pre-processing

### 2.1 Configuration

**USER ATTENTION NEEDED**: Edit the `NOTEBOOK_CONFIG` dictionary below to match your dataset.

In [2]:
# Code Chunk ID: CHUNK_2_1_1_A - NOTEBOOK CONFIGURATION
# ============================================================================
# USER CONFIGURATION - Edit ONLY this block for your dataset
# ============================================================================

NOTEBOOK_CONFIG = {
    # ========== REQUIRED: Dataset Settings ==========
    "data_file": "data/alzheimers_disease_data_processed.csv",  # Path to your CSV file
    "target_column": "Diagnosis",  # Target/outcome column name

    # ========== OPTIONAL: Dataset Metadata ==========
    "dataset_name": "Alzheimer's Dataset",  # Display name
    "dataset_identifier_override": None,  # Override auto-detected ID (or None)

    # ========== OPTIONAL: Column Configuration ==========
    # 17 categorical features: 15 binary flags + 2 multi-class (Ethnicity, EducationLevel)
    "categorical_columns": [
        "Gender",
        "Ethnicity",
        "EducationLevel",
        "Smoking",
        "FamilyHistoryAlzheimers",
        "CardiovascularDisease",
        "Diabetes",
        "Depression",
        "HeadInjury",
        "Hypertension",
        "MemoryComplaints",
        "BehavioralProblems",
        "Confusion",
        "Disorientation",
        "PersonalityChanges",
        "DifficultyCompletingTasks",
        "Forgetfulness",
    ],
    "task_type": "classification",

    # ========== OPTIONAL: Fairness Evaluation ==========
    # Protected attribute for fairness metrics (use cleaned column name after preprocessing).
    # Gender (binary 0/1) is the primary protected attribute.
    # Alternative: "ethnicity" (4 values, auto-binarized to majority vs rest).
    "protected_col": "gender",

    # ========== OPTIONAL: Data Subsetting ==========
    "use_row_subset": False,                      # 2149 rows - small enough to use all
    "sample_n": 500,                              # Number of rows to sample
    "sample_random_state": 42,                    # Random seed for reproducibility

    # ========== OPTIONAL: Missing Data Handling ==========
    "missing_strategy": "none",                   # No missing values in this dataset
    "mice_max_iter": 10,                          # Max iterations for MICE imputation

    # ========== OPTIONAL: Model Selection ==========
    "models_to_run": ["ctgan", "tvae", "copulagan", "ctabgan", "ctabganplus", "pategan", "medgan"],  # "all" or list like ["ctgan", "tvae"]
    # "models_to_run": ["ctgan", "tvae", "copulagan", "ctabgan", "ctabganplus", "ganeraid", "pategan", "medgan", "tabdiffusion", "great"],

    # ========== OPTIONAL: Collinearity Reduction ==========
    # Residual re-parameterization of near-deterministic column pairs
    # (|association| >= threshold). Disable to run without reduction.
    # Keep the engine in sync with multi-table-gen-compare/src/data/collinearity.py.
    "collinearity_enabled":   True,
    "collinearity_threshold": 0.975,
    "collinearity_overrides": {},     # {(col_a, col_b) alphabetical: {...}}

    # ========== OPTIONAL: Tuning Configuration ==========
    "tuning_mode": "smoke",                        # "smoke" (quick) | "full" (thorough)
    "n_trials_pilot": 2,                          # Trials for smoke testing / pilot phase
}

print("NOTEBOOK_CONFIG set successfully!")
print(f"  Data file: {NOTEBOOK_CONFIG['data_file']}")
print(f"  Target column: {NOTEBOOK_CONFIG['target_column']}")
print(f"  Protected column: {NOTEBOOK_CONFIG.get('protected_col', None)}")
print(f"  Models to run: {NOTEBOOK_CONFIG['models_to_run']}")
print(f"  Tuning mode: {NOTEBOOK_CONFIG['tuning_mode']}")

NOTEBOOK_CONFIG set successfully!
  Data file: data/alzheimers_disease_data_processed.csv
  Target column: Diagnosis
  Protected column: gender
  Models to run: ['ctgan', 'tvae', 'copulagan', 'ctabgan', 'ctabganplus', 'pategan', 'medgan']
  Tuning mode: smoke


### 2.2 Load and Preprocess Data

In [3]:
# Code Chunk ID: CHUNK_2_1_2_A - LOAD AND PREPROCESS DATA
# ============================================================================
# This uses the config-driven preprocessing pipeline
# ============================================================================

if not FRESH_START and 'checkpoint' in dir() and hasattr(checkpoint, 'exists') and checkpoint.exists('section_2'):
    saved = checkpoint.load('section_2')
    data = saved['data']
    original_data = saved['original_data']
    target_column = saved['target_column']
    DATASET_IDENTIFIER = saved['DATASET_IDENTIFIER']
    categorical_columns = saved['categorical_columns']
    metadata = saved['metadata']
    models_to_run = saved['models_to_run']
    RUN_MODE = saved['RUN_MODE']
    TARGET_COLUMN = target_column
    print("[RESUME] Loaded Section 2 from checkpoint")
else:
    # Load and preprocess data using the config
    (
        data,                  # Processed DataFrame
        original_data,         # Copy for reference
        target_column,         # Target column name (cleaned)
        DATASET_IDENTIFIER,    # Dataset identifier for results paths
        categorical_columns,   # List of categorical columns
        metadata               # Full preprocessing metadata
    ) = load_and_preprocess_from_config(NOTEBOOK_CONFIG)

    # Set aliases for backward compatibility with existing notebook cells
    TARGET_COLUMN = target_column

    # Get models to run based on config
    models_to_run = get_models_to_run(NOTEBOOK_CONFIG)

    # Set RUN_MODE for backward compatibility with Section 4 tuning cells
    RUN_MODE = "debug" if NOTEBOOK_CONFIG['tuning_mode'] == "smoke" else "full"

# Initialize checkpoint system (now that DATASET_IDENTIFIER is known)
checkpoint = SectionCheckpoint(DATASET_IDENTIFIER)

# If FRESH_START, wipe all checkpoints and optimization studies
if FRESH_START:
    n_removed = checkpoint.clear_all()
    print(f"[FRESH START] Cleared {n_removed} checkpoint(s) and optimization studies")

existing = checkpoint.list_checkpoints()
if existing:
    print(f"[CHECKPOINT] Found {len(existing)} existing checkpoint(s): {existing}")

# Save Section 2 checkpoint (idempotent - overwrites if exists)
if not checkpoint.exists('section_2'):
    checkpoint.save('section_2', {
        'data': data, 'original_data': original_data,
        'target_column': target_column, 'DATASET_IDENTIFIER': DATASET_IDENTIFIER,
        'categorical_columns': categorical_columns, 'metadata': metadata,
        'models_to_run': models_to_run, 'RUN_MODE': RUN_MODE,
    })

print()
print("=" * 60)
print("PREPROCESSING COMPLETE")
print("=" * 60)
print(f"  Dataset identifier: {DATASET_IDENTIFIER}")
print(f"  Processed shape: {data.shape}")
print(f"  Target column: {target_column}")
print(f"  Task type: {metadata['task_type']}")
print(f"  Categorical columns: {categorical_columns}")
print(f"  Models to run: {models_to_run}")
print(f"  RUN_MODE: {RUN_MODE}")
print(f"  Session: {SESSION_TIMESTAMP}")
print(f"  Results path: {get_results_path(DATASET_IDENTIFIER, 2)}")
print("=" * 60)

[LOAD] Loading data from: data/alzheimers_disease_data_processed.csv
[LOAD] Loaded 2149 rows, 33 columns
[PREPROCESS] Starting preprocessing pipeline
[PREPROCESS] Input shape: (2149, 33)
[PREPROCESS] Categorical columns: ['gender', 'ethnicity', 'educationlevel', 'smoking', 'familyhistoryalzheimers', 'cardiovasculardisease', 'diabetes', 'depression', 'headinjury', 'hypertension', 'memorycomplaints', 'behavioralproblems', 'confusion', 'disorientation', 'personalitychanges', 'difficultycompletingtasks', 'forgetfulness']
[PREPROCESS] Task type: classification
[PREPROCESS] Final shape: (2149, 33)
[PREPROCESS] Dataset identifier: alzheimers-disease-data-processed
[FLUSH] No previous studies found in results/alzheimers-disease-data-processed/optimization_studies — starting clean
[FRESH START] Cleared 9 checkpoint(s) and optimization studies

PREPROCESSING COMPLETE
  Dataset identifier: alzheimers-disease-data-processed
  Processed shape: (2149, 33)
  Target column: diagnosis
  Task type: clas

### 2.2b Collinearity Reduction

Near-deterministic column pairs (|association| ≥ `collinearity_threshold`) are
re-parameterized: one column is dropped and replaced with a residual, which the
generator learns alongside its surviving partner. Post-sample, the dropped
column is reconstructed from partner + residual — preserving near-perfect
relationships and their realistic imperfection.

The fit runs on the preprocessed frame. §3/§4/§5 train on the reduced schema.
§3.2 and §5.2 restore synthetic output to the full schema before evaluation and
emit `restoration_health.csv` for diagnosing per-pair regressions.

Review the decisions table below. To override a decision, edit
`NOTEBOOK_CONFIG['collinearity_overrides']` in §2.1 and re-run.


In [ ]:
# Code Chunk ID: CHUNK_2_2_B_COLLINEARITY - FIT REDUCER
# ============================================================================
# SECTION 2.2b - COLLINEARITY REDUCTION
# Residual re-parameterization of near-deterministic pairs. Context persists
# in COLLIN_CTX; §3.2 / §5.2 use it to restore dropped columns on synth data.
# ============================================================================

from src.data.collinearity import fit_collinearity_reducer, summarize_context

if not FRESH_START and checkpoint.exists('section_2.5'):
    saved = checkpoint.load('section_2.5')
    COLLIN_CTX = saved['collin_ctx']
    data = saved['data_reduced']
    data_full = saved.get('data_full', data.copy())  # fallback for old checkpoints
    categorical_columns = saved['categorical_columns']
    decisions = saved['decisions']
    print("[RESUME] Loaded §2.2b collinearity reducer from checkpoint")
else:
    COLLIN_CTX, data_reduced = fit_collinearity_reducer(
        data,
        threshold=NOTEBOOK_CONFIG['collinearity_threshold'],
        categorical_columns=categorical_columns,
        enabled=NOTEBOOK_CONFIG['collinearity_enabled'],
        overrides=NOTEBOOK_CONFIG.get('collinearity_overrides', {}),
    )
    decisions = summarize_context({DATASET_IDENTIFIER: COLLIN_CTX})
    data_full = data.copy()  # pre-reduction snapshot for §2.3 full-schema heatmap
    data = data_reduced
    # A dropped categorical column must be removed from the categorical list
    # before downstream calls pass it as discrete_columns.
    categorical_columns = [c for c in categorical_columns if c in data.columns]
    checkpoint.save('section_2.5', {
        'collin_ctx': COLLIN_CTX, 'data_reduced': data,
        'data_full': data_full,
        'categorical_columns': categorical_columns, 'decisions': decisions,
    })

print(f"[COLLIN] enabled={NOTEBOOK_CONFIG['collinearity_enabled']}, "
      f"threshold={NOTEBOOK_CONFIG['collinearity_threshold']}")
print(f"[COLLIN] Reduced schema: {data.shape}")
print(f"[COLLIN] Ops applied: {len(COLLIN_CTX.ops)}")
if COLLIN_CTX.ops:
    print(f"[COLLIN] Dropped: {COLLIN_CTX.dropped}")
    print(f"[COLLIN] Residual columns: {COLLIN_CTX.residual_columns}")

if not decisions.empty:
    print()
    print("Collinearity decisions (review before proceeding):")
    try:
        display(decisions)  # noqa: F821
    except NameError:
        print(decisions.to_string())
else:
    print(f"[COLLIN] No pairs at or above threshold "
          f"{NOTEBOOK_CONFIG['collinearity_threshold']}; no reduction applied")

# Flagged columns (union of kept + dropped across all decisions) for the
# §2.3 association heatmap — tick labels render in red for these columns.
COLLIN_FLAGGED = set()
if not decisions.empty:
    for col in ('col_a', 'col_b'):
        if col in decisions.columns:
            COLLIN_FLAGGED.update(decisions[col].dropna().astype(str).tolist())


### 2.3 Exploratory Data Analysis (EDA)

Comprehensive EDA with automated file export to results directory.

In [4]:
# Code Chunk ID: CHUNK_2_1_EDA - COMPREHENSIVE EDA
# ============================================================================
# Run comprehensive EDA with single function call.
# flagged_columns highlights collinearity-treated columns on the association
# heatmap (red, bold) so the reduction decisions are visually legible.
# ============================================================================

from src.data.eda import run_comprehensive_eda

eda_results = run_comprehensive_eda(
    data=data,
    target_column=target_column,
    dataset_identifier=DATASET_IDENTIFIER,
    dataset_name=NOTEBOOK_CONFIG.get('dataset_name'),
    categorical_columns=categorical_columns,
    verbose=True,
    flagged_columns=COLLIN_FLAGGED,
    full_data=data_full,
)

# Update categorical_columns from EDA results (in case auto-detected)
categorical_columns = eda_results['categorical_columns']

print(f"\nEDA Results Summary:")
print(f"  Files generated: {len(eda_results['files_generated'])}")
print(f"  Categorical columns: {categorical_columns}")
print(f"  Balance ratio: {eda_results.get('balance_ratio', 'N/A')}")


COMPREHENSIVE EXPLORATORY DATA ANALYSIS
Dataset: Alzheimer's Dataset
Target column: diagnosis
Results path: results/alzheimers-disease-data-processed/2026-03-25/Section-2

[1/6] Dataset Overview...
   Dataset Name.................. Alzheimer's Dataset
   Shape......................... 2149 rows x 33 columns
   Memory Usage.................. 0.54 MB
   Total Missing Values.......... 0
   Missing Percentage............ 0.00%
   Duplicate Rows................ 0
   Numeric Columns............... 33
   Categorical Columns........... 0

[2/6] Column Analysis...
   Saved: column_analysis.csv (33 columns)

[3/6] Target Variable Analysis...
   Saved: target_analysis.csv
   Saved: target_balance_metrics.csv
   Balance Ratio: 0.547 (Moderately Imbalanced)

[4/6] Feature Distributions...
   Saved: 6 distribution file(s) (32 features)

[5/6] Correlation Analysis...
   Saved: correlation_heatmap.png
   Saved: correlation_matrix.csv
   Saved: target_correlations.csv (32 features)

[6/6] Configuration

## 3 Demo All Models with Default Parameters

### 3.1 Batch Model Training

In [5]:
# Code Chunk ID: CHUNK_3_1_BATCH
# ============================================================================
# SECTION 3.1 - BATCH MODEL TRAINING
# Train all models configured in NOTEBOOK_CONFIG['models_to_run']
# ============================================================================

from src.models.batch_training import train_models_batch, extract_synthetic_data_to_globals

# Run batch training for all configured models (checkpoint resumes completed models)
training_results = train_models_batch(
    data=data,
    models_to_run=models_to_run,
    target_column=target_column,
    categorical_columns=categorical_columns,
    n_samples=len(data),
    random_state=42,
    verbose=True,
    checkpoint=checkpoint
)

# Extract synthetic_data_* variables for Section 3.2 compatibility
created_vars = extract_synthetic_data_to_globals(training_results, globals())
print(f"\nCreated variables: {created_vars}")

# Also create model_* variables for backward compatibility
for model_name, result in training_results.items():
    if result['status'] == 'success' and result.get('model') is not None:
        globals()[f'{model_name}_model'] = result['model']


BATCH MODEL TRAINING
Models to train: 7
Dataset shape: (2149, 33)
Target column: diagnosis
Samples to generate: 2149
GPU available: NVIDIA A10G (22.1 GB)
Device assignments:
  - CTGAN: cuda
  - TVAE: cuda
  - CopulaGAN: cuda
  - CTABGAN: cuda
  - CTABGAN+: cuda
  - PATE-GAN: cuda
  - MEDGAN: cuda


[1/7] Training CTGAN...
--------------------------------------------------
  Device: cuda
  Training CTGAN...


Gen. (-3.59) | Discrim. (0.05): 100%|██████████| 300/300 [00:44<00:00,  6.74it/s] 


  Generating 2149 synthetic samples...
  [OK] CTGAN completed in 53.40s
  Synthetic data shape: (2149, 33)

[2/7] Training TVAE...
--------------------------------------------------
  Device: cuda
  Training TVAE...
  Generating 2149 synthetic samples...
[OK] Target integrity functions loaded from src/data/target_integrity.py
  [OK] TVAE completed in 29.34s
  Synthetic data shape: (2149, 33)

[3/7] Training CopulaGAN...
--------------------------------------------------
  Device: cuda
  Training CopulaGAN...
  Generating 2149 synthetic samples...
  [OK] CopulaGAN completed in 48.69s
  Synthetic data shape: (2149, 33)

[4/7] Training CTABGAN...
--------------------------------------------------
  Device: cuda
  Training CTABGAN...


100%|██████████| 300/300 [01:07<00:00,  4.44it/s]


Finished training in 71.00996613502502  seconds.
  Generating 2149 synthetic samples...
  [OK] CTABGAN completed in 71.20s
  Synthetic data shape: (2149, 33)

[5/7] Training CTABGAN+...
--------------------------------------------------
  Device: cuda
  Training CTABGAN+...


100%|██████████| 400/400 [02:27<00:00,  2.70it/s]


Finished training in 151.34244966506958  seconds.
  Generating 2149 synthetic samples...
  [OK] CTABGAN+ completed in 151.54s
  Synthetic data shape: (2149, 33)

[6/7] Training PATE-GAN...
--------------------------------------------------
  Device: cuda
  Training PATE-GAN...
  Generating 2149 synthetic samples...
  [OK] PATE-GAN completed in 8.91s
  Synthetic data shape: (2149, 33)

[7/7] Training MEDGAN...
--------------------------------------------------
  Device: cuda
  Training MEDGAN...
  Generating 2149 synthetic samples...
  [OK] MEDGAN completed in 24.37s
  Synthetic data shape: (2149, 33)

BATCH TRAINING SUMMARY
Total models: 7
Successful: 7
Failed: 0

Successful models:
  - CTGAN: 53.40s
  - TVAE: 29.34s
  - CopulaGAN: 48.69s
  - CTABGAN: 71.20s
  - CTABGAN+: 151.54s
  - PATE-GAN: 8.91s
  - MEDGAN: 24.37s

Created variables: ['synthetic_data_ctgan', 'synthetic_data_tvae', 'synthetic_data_copulagan', 'synthetic_data_ctabgan', 'synthetic_data_ctabganplus', 'synthetic_data_pa

### 3.2 Batch Evaluation

In [6]:
# Code Chunk ID: CHUNK_3_2_0_A
# ============================================================================
# SECTION 3.2 - BATCH EVALUATION FOR ALL TRAINED MODELS
# Collinearity: restore dropped columns on synthetic_data_* globals so §3
# metrics reflect the full schema (comparable to §5), then emit
# restoration_health.csv for per-pair residual-preservation diagnostics.
# ============================================================================
import os

print("SECTION 3.2 - COMPREHENSIVE BATCH EVALUATION")
print("=" * 60)

# ---- Collinearity restore before evaluation ----
if 'COLLIN_CTX' in globals() and COLLIN_CTX.ops:
    from src.data.collinearity import restore_dropped
    restored_names = []
    for name in list(globals()):
        if name.startswith('synthetic_data_') and isinstance(globals()[name], pd.DataFrame):
            globals()[name] = restore_dropped(globals()[name], COLLIN_CTX)
            restored_names.append(name)
    if restored_names:
        print(f"[COLLIN] Restored dropped columns on "
              f"{len(restored_names)} synthetic dataset(s)")

if checkpoint.exists('section_3.2'):
    section3_results = checkpoint.load('section_3.2')['results']
    print("[RESUME] Loaded Section 3.2 evaluation from checkpoint")
else:
    section3_results = evaluate_all_available_models(
        section_number=3,
        scope=globals(),
        models_to_evaluate=None,
        real_data=None,
        target_col=None,
        protected_col=NOTEBOOK_CONFIG.get("protected_col")
    )
    checkpoint.save('section_3.2', {'results': section3_results})

# ---- Emit restoration_health.csv (per-model residual-preservation diagnostic) ----
if 'COLLIN_CTX' in globals() and COLLIN_CTX.ops:
    from src.data.collinearity import restoration_health
    from src.utils.paths import get_results_path
    health_rows = []
    for name in [n for n in globals() if n.startswith('synthetic_data_')]:
        synth_df = globals()[name]
        if not isinstance(synth_df, pd.DataFrame):
            continue
        h = restoration_health({'t': original_data}, {'t': synth_df}, {'t': COLLIN_CTX})
        if not h.empty:
            h = h.drop(columns=['table'], errors='ignore')
            h['model'] = name.replace('synthetic_data_', '')
            health_rows.append(h)
    if health_rows:
        all_health = pd.concat(health_rows, ignore_index=True)
        results_dir = get_results_path(DATASET_IDENTIFIER, 3)
        os.makedirs(results_dir, exist_ok=True)
        out_path = os.path.join(results_dir, 'restoration_health.csv')
        all_health.to_csv(out_path, index=False)
        print(f"[COLLIN] restoration_health.csv -> {out_path} "
              f"({len(all_health)} rows)")

if section3_results:
    print(f"\nSECTION 3 BATCH EVALUATION COMPLETED!")
    print(f"Evaluated {len(section3_results)} models successfully")

    # Show ranking by quality score
    best_models = []
    for model_name, results in section3_results.items():
        if 'error' not in results:
            quality_score = results.get('overall_quality_score', 0)
            best_models.append((model_name, quality_score))

    if best_models:
        best_models.sort(key=lambda x: x[1], reverse=True)
        print(f"\nRANKING BY QUALITY SCORE:")
        for i, (model, score) in enumerate(best_models, 1):
            print(f"   {i}. {model}: {score:.3f}")
else:
    print("\nNo models available for evaluation")


SECTION 3.2 - COMPREHENSIVE BATCH EVALUATION
[OK] Batch evaluation functions loaded from src/evaluation/batch.py
[SEARCH] UNIFIED BATCH EVALUATION - SECTION 3
[INFO] Dataset: alzheimers-disease-data-processed
[INFO] Target column: diagnosis
[INFO] Protected column: gender
[INFO] MIA evaluation: OFF
[INFO] Variable pattern: standard
[INFO] Found 7 trained models:
   [OK] CTGAN
   [OK] CTABGAN
   [OK] CTABGANPLUS
   [OK] CopulaGAN
   [OK] TVAE
   [OK] MEDGAN
   [OK] PATEGAN

==================== EVALUATING CTGAN ====================
[SEARCH] CTGAN - COMPREHENSIVE DATA QUALITY EVALUATION
[FOLDER] Output directory: results/alzheimers-disease-data-processed/2026-03-25/Section-3/CTGAN

[1] STATISTICAL SIMILARITY
------------------------------
   [CHART] Average Statistical Similarity: 0.820

[2] PCA COMPARISON ANALYSIS WITH OUTCOME COLOR-CODING
--------------------------------------------------
   [CHART] PCA comparison plot saved: pca_comparison_with_outcome.png
   [CHART] PCA Overall Simil

## 4 Staged Hyperparameter Optimization

This section uses a staged approach to hyperparameter optimization:

- **Smoke mode** (`tuning_mode="smoke"`): Runs 10 pilot trials per model, then displays
  time-budget recommendations (how many trials fit in 1 hour, how long 20 trials would take).
  Section 5 uses the pilot results directly.
- **Full mode** (`tuning_mode="full"`): Runs a pilot phase, displays time estimates, then
  presents three continuation strategies in cell 4.3.  The user reviews the estimates and
  **uncomments one option** before running the cell.

### 4.1 Configuration

Create the `StagedOptimizationManager`.  `TUNING_MODE` and `PILOT_TRIALS` are derived
from `NOTEBOOK_CONFIG` so the same code works for both smoke and full runs.

In [7]:
# Code Chunk ID: CHUNK_4_1_CONFIG
# ============================================================================
# SECTION 4.1 - STAGED OPTIMIZATION CONFIGURATION
# ============================================================================

from src.models.staged_optimization import (
    StagedOptimizationManager,
    StagedOptimizationConfig,
    flush_previous_runs
)

# Flush optimization studies if FRESH_START is set
if FRESH_START:
    flush_previous_runs(DATASET_IDENTIFIER)

# Derive tuning mode and pilot size from NOTEBOOK_CONFIG
TUNING_MODE = NOTEBOOK_CONFIG.get("tuning_mode", "smoke")
PILOT_TRIALS = 10 if TUNING_MODE == "smoke" else NOTEBOOK_CONFIG.get("n_trials_pilot", 10)

# Configure staged optimization
staged_config = StagedOptimizationConfig(
    pilot_trials=PILOT_TRIALS,
    verbose_level=1,           # 0=silent, 1=concise, 2=detailed
    persistence_dir=f"results/{DATASET_IDENTIFIER}/optimization_studies",
    run_mode=RUN_MODE,         # "debug" or "full"
    random_state=42,
    continue_on_error=True
)

# Create the manager
manager = StagedOptimizationManager(
    data=data,
    target_column=target_column,
    categorical_columns=categorical_columns,
    dataset_identifier=DATASET_IDENTIFIER,
    config=staged_config
)

print("Staged Optimization Manager created!")
print(f"  Tuning mode: {TUNING_MODE}")
print(f"  Pilot trials: {staged_config.pilot_trials}")
print(f"  Run mode: {staged_config.run_mode}")
print(f"  Persistence dir: {staged_config.persistence_dir}")
print(f"  FRESH_START: {FRESH_START}")

[FLUSH] No previous studies found in results/alzheimers-disease-data-processed/optimization_studies — starting clean
Staged Optimization Manager created!
  Tuning mode: smoke
  Pilot trials: 10
  Run mode: debug
  Persistence dir: results/alzheimers-disease-data-processed/optimization_studies
  FRESH_START: True


### 4.2 Run Pilot Phase

Run a pilot phase to establish time estimates.

- **Smoke mode**: 10 trials + smoke recommendations table (trials in 1 hr, time for 20 trials).
- **Full mode**: Pilot trials + time estimates for planning continuation.

In [ ]:
# Code Chunk ID: CHUNK_4_2_PILOT
# ============================================================================
# SECTION 4.2 - RUN PILOT PHASE
# ============================================================================

# Run pilot phase (uses PILOT_TRIALS from cell 4.1)
pilot_states = manager.run_pilot(
    models_to_run=models_to_run
)

# Display time estimates
print("\n" + "="*60)
print("PILOT PHASE COMPLETE - TIME ESTIMATES")
print("="*60)

time_estimates = manager.get_time_estimates()
if len(time_estimates) > 0:
    print(time_estimates.to_string(index=False))
else:
    print("No time estimates available")

# Show best scores so far
print("\nBest scores after pilot:")
for model_name, state in pilot_states.items():
    print(f"  {model_name}: {state.best_score:.4f} ({state.total_trials} trials)")

# Smoke mode: show budget recommendations
if TUNING_MODE == "smoke":
    print("\n" + "="*60)
    print("SMOKE MODE - TIME BUDGET RECOMMENDATIONS")
    print("="*60)
    smoke_recs = manager.get_smoke_recommendations()
    print(smoke_recs.to_string(index=False))
    print("\nTo run a full optimization, set tuning_mode='full' in NOTEBOOK_CONFIG")
    print("and use the recommended_pilot column to guide n_trials_pilot.")

[I 2026-03-25 15:04:35,218] A new study created in memory with name: ctgan_hpo_alzheimers-disease-data-processed



STAGED OPTIMIZATION - PILOT PHASE
Models to optimize: 7
Pilot trials per model: 10
Dataset identifier: alzheimers-disease-data-processed


[PILOT] Optimizing CTGAN...
--------------------------------------------------


Gen. (-3.74) | Discrim. (0.09): 100%|██████████| 300/300 [01:27<00:00,  3.42it/s] 


[TARGET] Enhanced objective function using target column: 'diagnosis'
[OK] Similarity Analysis: 33/33 valid metrics, Average: 0.6662
[OK] TRTS Evaluation: 2 scenarios, Average: 0.5859
[CHART] Combined Score: 0.6341 (Similarity: 0.6662, Accuracy: 0.5859)
[ctgan] Trial 1: Combined Score: 0.6341 (Similarity: 0.6662, Accuracy: 0.5859) Best Score so far: 0.6341


Gen. (-4.62) | Discrim. (0.03): 100%|██████████| 400/400 [01:55<00:00,  3.46it/s] 


...

Gen. (-5.17) | Discrim. (-0.21): 100%|██████████| 400/400 [01:54<00:00,  3.48it/s]


[TARGET] Enhanced objective function using target column: 'diagnosis'
[OK] Similarity Analysis: 33/33 valid metrics, Average: 0.6538
[OK] TRTS Evaluation: 2 scenarios, Average: 0.4923
[CHART] Combined Score: 0.5892 (Similarity: 0.6538, Accuracy: 0.4923)
[ctgan] Trial 7: Combined Score: 0.5892 (Similarity: 0.6538, Accuracy: 0.4923) Best Score so far: 0.6540


Gen. (-4.60) | Discrim. (-0.10): 100%|██████████| 350/350 [01:39<00:00,  3.50it/s]


[TARGET] Enhanced objective function using target column: 'diagnosis'
[OK] Similarity Analysis: 33/33 valid metrics, Average: 0.6196
[OK] TRTS Evaluation: 2 scenarios, Average: 0.5903
[CHART] Combined Score: 0.6078 (Similarity: 0.6196, Accuracy: 0.5903)
[ctgan] Trial 8: Combined Score: 0.6078 (Similarity: 0.6196, Accuracy: 0.5903) Best Score so far: 0.6540


Gen. (-3.08) | Discrim. (-0.13):  57%|█████▋    | 201/350 [00:55<00:41,  3.56it/s]

In [ ]:
# ============================================================================
# DIMINISHING RETURNS ANALYSIS
# ============================================================================
print("DIMINISHING RETURNS ANALYSIS")
print("="*60)

convergence_report = manager.get_diminishing_returns_report()

if len(convergence_report) > 0:
    print(convergence_report.to_string(index=False))
    
    print("\nInterpretation:")
    print("-" * 40)
    for _, row in convergence_report.iterrows():
        print(f"  {row['model']}: {row['recommendation']}")
        if row['has_plateaued']:
            print(f"    -> Model has plateaued at score {row['best_score']:.4f}")
        else:
            print(f"    -> Improvement rate: {row['improvement_rate']:.4f}")
else:
    print("No convergence data available")

### 4.3 Continuation (Full Mode Only)

**Full mode only** - Review the pilot results and time estimates above, then
uncomment **one** of the three options below and adjust the values before running.

- **Option (i)**: Common trial count for all models
- **Option (ii)**: Per-model trial counts
- **Option (iii)**: Time-based budget (minutes per model)

Models not in `models_to_run` are silently ignored, so listing all 8 is safe.

In [ ]:
# Code Chunk ID: CHUNK_4_3_CONTINUE
# ============================================================================
# SECTION 4.3 - CONTINUATION (Full mode only - choose ONE option)
# ============================================================================

if TUNING_MODE == "smoke":
    print("[SMOKE MODE] Skipping continuation - using pilot results for Section 5.")
else:
    # ---- Uncomment ONE option below, adjust values, then run this cell ----

    # OPTION (i): Common trials for all models
    # continuation_states = manager.continue_optimization(additional_trials=30)

    # OPTION (ii): Per-model trials - adjust counts per model
    # continuation_states = manager.continue_optimization(
    #     trials_per_model={
    #         'ctgan': 30,
    #         'tvae': 30,
    #         'copulagan': 30,
    #         'ctabgan': 30,
    #         'ctabganplus': 30,
    #         'ganeraid': 30,
    #         'pategan': 30,
    #         'medgan': 30,
    #     }
    # )

    # OPTION (iii): Time-based budget - minutes per model
    continuation_states = manager.continue_optimization(
        time_budget_minutes={
     #       'ctgan': 20,
            'tvae': 10,
     #       'copulagan': 60,
     #       'ctabgan': 60,
            'ctabganplus': 10,
      #      'ganeraid': 60,
            'pategan': 10,
            'medgan': 10,
        }
    )

    print("[FULL MODE] Uncomment one option above and re-run this cell.")

In [ ]:
# ============================================================================
# DIMINISHING RETURNS ANALYSIS (post-continuation)
# ============================================================================

if TUNING_MODE == "full":
    print("DIMINISHING RETURNS ANALYSIS")
    print("="*60)

    convergence_report = manager.get_diminishing_returns_report()

    if len(convergence_report) > 0:
        print(convergence_report.to_string(index=False))

        print("\nInterpretation:")
        print("-" * 40)
        for _, row in convergence_report.iterrows():
            print(f"  {row['model']}: {row['recommendation']}")
            if row['has_plateaued']:
                print(f"    -> Model has plateaued at score {row['best_score']:.4f}")
            else:
                print(f"    -> Improvement rate: {row['improvement_rate']:.4f}")
    else:
        print("No convergence data available")
else:
    print("[SMOKE MODE] Skipping post-continuation analysis.")

### 4.5 Additional Batches (Full Mode, Optional)

If the diminishing returns analysis suggests continuing, uncomment one option below.
You can re-run this cell multiple times.

In [ ]:
# Code Chunk ID: CHUNK_4_5_ADDITIONAL
# ============================================================================
# SECTION 4.5 - ADDITIONAL BATCHES (Full mode only - optional, can repeat)
# ============================================================================

if TUNING_MODE == "smoke":
    print("[SMOKE MODE] Skipping additional batches.")
else:
    # ---- Uncomment ONE option below, adjust values, then run this cell ----

    # OPTION (i): Common trials for all models
    # additional_states = manager.continue_optimization(additional_trials=20)

    # OPTION (ii): Per-model trials - adjust counts per model
    # additional_states = manager.continue_optimization(
    #     trials_per_model={
    #         'ctgan': 20,
    #         'tvae': 20,
    #         'copulagan': 20,
    #         'ctabgan': 20,
    #         'ctabganplus': 20,
    #         'ganeraid': 20,
    #         'pategan': 20,
    #         'medgan': 20,
    #     }
    # )

    # OPTION (iii): Time-based budget - minutes per model
    # additional_states = manager.continue_optimization(
    #     time_budget_minutes={
    #         'ctgan': 30,
    #         'tvae': 30,
    #         'copulagan': 30,
    #         'ctabgan': 30,
    #         'ctabganplus': 30,
    #         'ganeraid': 30,
    #         'pategan': 30,
    #         'medgan': 30,
    #     }
    # )

    # print("\nUpdated convergence report:")
    # print(manager.get_diminishing_returns_report().to_string(index=False))

    print("Cell skipped - uncomment an option above to run additional batches")

### 4.6 Save Best Parameters

In [ ]:
# Code Chunk ID: CHUNK_4_6_SAVE
# ============================================================================
# SECTION 4.6 - SAVE BEST PARAMETERS TO CSV
# ============================================================================

from src.utils.parameters import save_best_parameters_to_csv

# Extract studies from manager to globals for saving
for model_name, state in manager.model_states.items():
    if state.study is not None:
        globals()[f"{model_name}_study"] = state.study

# Save all best parameters to CSV for Section 5
save_result = save_best_parameters_to_csv(
    scope=globals(),
    section_number=4,
    dataset_identifier=DATASET_IDENTIFIER
)

print(f"\nParameters saved: {save_result['success']}")
print(f"Files: {save_result.get('files_saved', [])}")

## 5 Final Model Comparison with Best Parameters

### 5.1 Train All Models with Best Parameters from Section 4

In [ ]:
# Code Chunk ID: CHUNK_5_1_BATCH
# ============================================================================
# SECTION 5.1 - BATCH TRAINING WITH BEST PARAMETERS
# ============================================================================

from src.models.batch_training import train_models_batch_with_best_params

# Train all models with best parameters from Section 4 (checkpoint resumes completed models)
section5_results = train_models_batch_with_best_params(
    data=data,
    target_column=target_column,
    models_to_run=models_to_run,
    categorical_columns=categorical_columns,
    n_samples=len(data),
    random_state=42,
    section_number=4,
    dataset_identifier=DATASET_IDENTIFIER,
    scope=globals(),
    verbose=True,
    checkpoint=checkpoint
)

# Show summary of created variables
final_vars = [k for k in globals().keys() if k.endswith('_final')]
print(f"\nFinal synthetic data variables: {sorted(final_vars)}")

### 5.2 Batch Evaluation of Optimized Models

In [ ]:
# Code Chunk ID: CHUNK_5_2_0_A
# ============================================================================
# SECTION 5.2 - OPTIMIZED MODELS BATCH EVALUATION
# Collinearity: restore dropped columns on synthetic_*_final globals before
# evaluation. Emits restoration_health.csv alongside the §5 scorecards.
# ============================================================================
import os

print("SECTION 5.2 - OPTIMIZED MODELS BATCH EVALUATION")
print("=" * 80)

from setup import evaluate_section5_optimized_models

# ---- Collinearity restore before evaluation ----
if 'COLLIN_CTX' in globals() and COLLIN_CTX.ops:
    from src.data.collinearity import restore_dropped
    restored_names = []
    for name in list(globals()):
        if name.startswith('synthetic_') and name.endswith('_final') \
                and isinstance(globals()[name], pd.DataFrame):
            globals()[name] = restore_dropped(globals()[name], COLLIN_CTX)
            restored_names.append(name)
    if restored_names:
        print(f"[COLLIN] Restored dropped columns on "
              f"{len(restored_names)} final synthetic dataset(s)")

if checkpoint.exists('section_5.2'):
    section5_batch_results = checkpoint.load('section_5.2')['results']
    print("[RESUME] Loaded Section 5.2 evaluation from checkpoint")
    print(f"Models processed: {section5_batch_results['models_processed']}")
    print(f"Results exported to: {section5_batch_results['results_dir']}")
else:
    try:
        section5_batch_results = evaluate_section5_optimized_models(
            section_number=5,
            scope=globals(),
            target_column=TARGET_COLUMN,
            protected_col=NOTEBOOK_CONFIG.get("protected_col"),
            compute_mia=True
        )
        checkpoint.save('section_5.2', {'results': section5_batch_results})

        print("\n" + "="*80)
        print("SECTION 5.2 OPTIMIZED MODELS BATCH EVALUATION COMPLETED!")
        print("="*80)
        print(f"Models processed: {section5_batch_results['models_processed']}")
        print(f"Results exported to: {section5_batch_results['results_dir']}")

    except Exception as e:
        print(f"Section 5.2 batch evaluation failed: {e}")
        import traceback
        traceback.print_exc()

# ---- Emit restoration_health.csv (per-model residual-preservation diagnostic) ----
if 'COLLIN_CTX' in globals() and COLLIN_CTX.ops:
    from src.data.collinearity import restoration_health
    from src.utils.paths import get_results_path
    health_rows = []
    for name in [n for n in globals() if n.startswith('synthetic_') and n.endswith('_final')]:
        synth_df = globals()[name]
        if not isinstance(synth_df, pd.DataFrame):
            continue
        h = restoration_health({'t': original_data}, {'t': synth_df}, {'t': COLLIN_CTX})
        if not h.empty:
            h = h.drop(columns=['table'], errors='ignore')
            h['model'] = name[len('synthetic_'):-len('_final')]
            health_rows.append(h)
    if health_rows:
        all_health = pd.concat(health_rows, ignore_index=True)
        results_dir = get_results_path(DATASET_IDENTIFIER, 5)
        os.makedirs(results_dir, exist_ok=True)
        out_path = os.path.join(results_dir, 'restoration_health.csv')
        all_health.to_csv(out_path, index=False)
        print(f"[COLLIN] restoration_health.csv -> {out_path} "
              f"({len(all_health)} rows)")


### 5.3 Final Summary

In [ ]:
# Code Chunk ID: CHUNK_5_3_SUMMARY
# ============================================================================
# SECTION 5.3 - FINAL SUMMARY
# ============================================================================

print("=" * 80)
print("SYNTHETIC DATA GENERATION - FINAL SUMMARY")
print("=" * 80)
print(f"\nDataset: {DATASET_IDENTIFIER}")
print(f"Session: {SESSION_TIMESTAMP}")
print(f"\nResults directories:")
print(f"  Section 2 (EDA): {get_results_path(DATASET_IDENTIFIER, 2)}")
print(f"  Section 3 (Demo): {get_results_path(DATASET_IDENTIFIER, 3)}")
print(f"  Section 4 (HPO): {get_results_path(DATASET_IDENTIFIER, 4)}")
print(f"  Section 5 (Final): {get_results_path(DATASET_IDENTIFIER, 5)}")

# Show staged optimization summary
if 'manager' in dir() and manager is not None:
    print(f"\nStaged Optimization Summary:")
    print("-" * 60)
    summary = manager.get_best_params_summary()
    if len(summary) > 0:
        print(summary[['model', 'best_score', 'total_trials', 'status']].to_string(index=False))

# Show final model performance
if 'section5_results' in dir() and section5_results:
    print(f"\nFinal Model Performance (with optimized parameters):")
    print("-" * 60)
    
    scores = []
    for model_name, result in section5_results.items():
        if result['status'] == 'success':
            score = result.get('objective_score', 0)
            time_taken = result.get('training_time', 0)
            scores.append((model_name, score, time_taken))
    
    # Sort by score descending
    scores.sort(key=lambda x: x[1], reverse=True)
    
    for i, (model, score, time_taken) in enumerate(scores, 1):
        print(f"  {i}. {model.upper()}: score={score:.4f}, time={time_taken:.1f}s")
    
    if scores:
        best_model = scores[0][0]
        print(f"\nBest performing model: {best_model.upper()}")
        print(f"Best synthetic data variable: synthetic_{best_model}_final")

print("\n" + "=" * 80)
print("NOTEBOOK COMPLETE")
print("=" * 80)